# Statistical Arbitrage Analysis

Intraday mean-reversion pairs trading system for US equities.

**Workflow:**
1. Data must be fetched first: `python scripts/fetch_data.py`
2. Preprocess data (resample to multiple timeframes)
3. Discover cointegrated pairs on daily data
4. Generate signals on intraday data
5. Backtest and analyze performance

## 1. Setup & Imports

In [1]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import numpy as np

from utils.config import CONFIG, get_all_tickers, get_all_sectors
from analysis.preprocessing import (
    preprocess_all_tickers,
    load_processed,
    load_raw,
)
from analysis.cointegration import (
    find_cointegrated_pairs,
)
print(f"Polars version: {pl.__version__}")
print(f"Project root:   {project_root}")
print(f"Data directory:  {CONFIG['data_dir']}")

Polars version: 1.38.1
Project root:   /Users/erevtsov/dev/stat-arb
Data directory:  /Users/erevtsov/dev/stat-arb/data


## 2. Data Loading & Preprocessing

In [2]:
# Preprocess all raw 1-min data into multiple timeframes
# This reads from data/raw/1min/ and writes to data/processed/{timeframe}/
# Only need to run once (or when raw data changes)

summary = preprocess_all_tickers()
print(summary)

Preprocessing:   0%|          | 0/100 [00:00<?, ?it/s]

Preprocessing: 100%|██████████| 100/100 [00:04<00:00, 24.99it/s]

shape: (100, 6)
┌────────┬───────┬────────┬───────┬───────┬───────┐
│ ticker ┆ daily ┆ 1min   ┆ 5min  ┆ 15min ┆ 1hour │
│ ---    ┆ ---   ┆ ---    ┆ ---   ┆ ---   ┆ ---   │
│ str    ┆ i64   ┆ i64    ┆ i64   ┆ i64   ┆ i64   │
╞════════╪═══════╪════════╪═══════╪═══════╪═══════╡
│ AAPL   ┆ 1003  ┆ 184209 ┆ 39183 ┆ 13541 ┆ 4016  │
│ ABBV   ┆ 1003  ┆ 89697  ┆ 26006 ┆ 10938 ┆ 3616  │
│ ABT    ┆ 1003  ┆ 77520  ┆ 20676 ┆ 9009  ┆ 3226  │
│ ADBE   ┆ 1003  ┆ 120779 ┆ 31862 ┆ 11963 ┆ 3708  │
│ AEP    ┆ 1003  ┆ 73665  ┆ 18597 ┆ 8051  ┆ 2925  │
│ …      ┆ …     ┆ …      ┆ …     ┆ …     ┆ …     │
│ VLO    ┆ 1003  ┆ 78632  ┆ 21306 ┆ 9191  ┆ 3188  │
│ VZ     ┆ 1003  ┆ 120119 ┆ 32724 ┆ 12566 ┆ 3865  │
│ WFC    ┆ 1003  ┆ 92507  ┆ 25935 ┆ 10784 ┆ 3618  │
│ WMT    ┆ 1003  ┆ 115475 ┆ 30705 ┆ 11807 ┆ 3716  │
│ XOM    ┆ 1003  ┆ 128724 ┆ 34358 ┆ 12770 ┆ 3905  │
└────────┴───────┴────────┴───────┴───────┴───────┘


In [3]:
# Show available tickers and data summary
raw_dir = Path(CONFIG["raw_dir"])
available_tickers = sorted(p.stem for p in raw_dir.glob("*.parquet"))
print(f"Available tickers: {len(available_tickers)}")
print(available_tickers)

# Quick data quality check on first ticker
if available_tickers:
    sample_ticker = available_tickers[0]
    df_sample = load_processed(sample_ticker, "daily")
    print(f"\n--- {sample_ticker} daily data ---")
    print(f"Shape: {df_sample.shape}")
    print(f"Date range: {df_sample['timestamp'].min()} to {df_sample['timestamp'].max()}")
    print(df_sample.describe())

Available tickers: 100
['AAPL', 'ABBV', 'ABT', 'ADBE', 'AEP', 'AMAT', 'AMD', 'AMGN', 'AMT', 'AMZN', 'AVGO', 'BA', 'BAC', 'BKNG', 'BLK', 'BMY', 'C', 'CAT', 'CCI', 'CHTR', 'CL', 'CMCSA', 'CMG', 'COP', 'COST', 'CRM', 'CVX', 'D', 'DE', 'DIS', 'DUK', 'EA', 'EOG', 'EQIX', 'EXC', 'GE', 'GOOG', 'GOOGL', 'GS', 'HD', 'HON', 'INTC', 'JNJ', 'JPM', 'KHC', 'KLAC', 'KO', 'LLY', 'LMT', 'LOW', 'LRCX', 'MCD', 'MCHP', 'MDLZ', 'META', 'MMM', 'MPC', 'MRK', 'MS', 'MSFT', 'MU', 'NEE', 'NFLX', 'NKE', 'NVDA', 'O', 'ON', 'ORCL', 'OXY', 'PEP', 'PFE', 'PG', 'PLD', 'PM', 'PNC', 'PSA', 'PSX', 'QCOM', 'RTX', 'SBUX', 'SCHW', 'SLB', 'SO', 'SPG', 'SRE', 'T', 'TJX', 'TMO', 'TMUS', 'TSLA', 'TXN', 'UNH', 'UNP', 'UPS', 'USB', 'VLO', 'VZ', 'WFC', 'WMT', 'XOM']

--- AAPL daily data ---
Shape: (1003, 6)
Date range: 2022-01-03 00:00:00 to 2025-12-31 00:00:00
shape: (9, 7)
┌────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬─────────────────┐
│ statistic  ┆ open       ┆ high       ┆ low        ┆ close

In [4]:
# Verify data quality: check for missing bars and outliers
quality_report = []
for ticker in available_tickers:
    try:
        df = load_processed(ticker, "1min")
        daily = load_processed(ticker, "daily")
        quality_report.append({
            "ticker": ticker,
            "1min_bars": len(df),
            "daily_bars": len(daily),
            "min_close": df["close"].min(),
            "max_close": df["close"].max(),
            "null_count": df.null_count().sum_horizontal()[0],
        })
    except Exception as e:
        print(f"Error loading {ticker}: {e}")

quality_df = pl.DataFrame(quality_report)
print(quality_df)

shape: (100, 6)
┌────────┬───────────┬────────────┬───────────┬───────────┬────────────┐
│ ticker ┆ 1min_bars ┆ daily_bars ┆ min_close ┆ max_close ┆ null_count │
│ ---    ┆ ---       ┆ ---        ┆ ---       ┆ ---       ┆ ---        │
│ str    ┆ i64       ┆ i64        ┆ f64       ┆ f64       ┆ i64        │
╞════════╪═══════════╪════════════╪═══════════╪═══════════╪════════════╡
│ AAPL   ┆ 184209    ┆ 1003       ┆ 124.44    ┆ 259.9299  ┆ 0          │
│ ABBV   ┆ 89697     ┆ 1003       ┆ 131.01    ┆ 207.26    ┆ 0          │
│ ABT    ┆ 77520     ┆ 1003       ┆ 89.8      ┆ 122.0     ┆ 0          │
│ ADBE   ┆ 120779    ┆ 1003       ┆ 318.84    ┆ 636.32    ┆ 0          │
│ AEP    ┆ 73665     ┆ 1003       ┆ 69.43     ┆ 105.21    ┆ 0          │
│ …      ┆ …         ┆ …          ┆ …         ┆ …         ┆ …          │
│ VLO    ┆ 78632     ┆ 1003       ┆ 104.31    ┆ 200.0     ┆ 0          │
│ VZ     ┆ 120119    ┆ 1003       ┆ 30.1699   ┆ 45.26     ┆ 0          │
│ WFC    ┆ 92507     ┆ 1003       ┆

## 3. Pair Discovery (Cointegration Analysis)

In [5]:
# Run cointegration analysis on daily data
coint_pairs = find_cointegrated_pairs(timeframe='15min', start_date='2023-01-01', end_date='2023-06-30')
print(f"Found {len(coint_pairs)} cointegrated pairs")
coint_pairs = coint_pairs.sort(by='p_value')
coint_pairs.sort(by='p_value')

Testing cointegration: 100%|██████████| 411/411 [00:00<00:00, 807.31it/s]

Found 104 cointegrated pairs


ticker_a,ticker_b,hedge_ratio,adf_stat,p_value,half_life,sector,timeframe,formation_start,formation_end,n_observations
str,str,f64,f64,f64,f64,str,str,str,str,i64
"""MSFT""","""META""",0.730877,-4.771446,0.000062,56.632644,"""Technology""","""15min""","""2023-01-01""","""2023-06-30""",3126
"""AAPL""","""META""",0.344049,-4.584524,0.000138,51.261192,"""Technology""","""15min""","""2023-01-01""","""2023-06-30""",3169
"""SCHW""","""PNC""",0.713731,-4.58163,0.00014,23.888793,"""Financials""","""15min""","""2023-01-01""","""2023-06-30""",1607
"""PFE""","""TMO""",0.081915,-4.517489,0.000183,41.96038,"""Healthcare""","""15min""","""2023-01-01""","""2023-06-30""",1279
"""BA""","""MMM""",0.155378,-4.487444,0.000207,35.123404,"""Industrials""","""15min""","""2023-01-01""","""2023-06-30""",1874
…,…,…,…,…,…,…,…,…,…,…
"""NEE""","""AEP""",0.518703,-2.920443,0.043022,65.389326,"""Utilities""","""15min""","""2023-01-01""","""2023-06-30""",1326
"""TXN""","""MU""",-0.044327,-2.907825,0.044446,61.397157,"""Semiconductors""","""15min""","""2023-01-01""","""2023-06-30""",1591
"""USB""","""PNC""",0.391196,-2.888014,0.04676,50.738206,"""Financials""","""15min""","""2023-01-01""","""2023-06-30""",1566


In [6]:
# Visualize top pairs: scatter plots of prices
if len(coint_pairs) >= 3:
    n_plots = min(6, len(coint_pairs))
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=[f"{r['ticker_a']}/{r['ticker_b']} (p={r['p_value']:.4f})"
                        for r in coint_pairs.head(n_plots).iter_rows(named=True)]
    )
    for idx, row in enumerate(coint_pairs.head(n_plots).iter_rows(named=True)):
        r, c = divmod(idx, 3)
        df_a = load_processed(row["ticker_a"], "daily")
        df_b = load_processed(row["ticker_b"], "daily")
        merged = df_a.select(["timestamp", "close"]).rename({"close": "A"}).join(
            df_b.select(["timestamp", "close"]).rename({"close": "B"}),
            on="timestamp", how="inner"
        )
        fig.add_trace(
            go.Scatter(x=merged["A"].to_list(), y=merged["B"].to_list(),
                       mode="markers", marker=dict(size=3), showlegend=False),
            row=r+1, col=c+1
        )
    fig.update_layout(height=600, title_text="Price Scatter Plots - Top Cointegrated Pairs")
    fig.show()